In [1]:
import pandas as pd

df = pd.read_json("data/llm_hallucination_detection_dataset.jsonl", lines=True)
df.shape

(5000, 15)

In [2]:
df.head(3)

,id,prompt,context,response,label,hallucination_type,domain,task_type,evidence_available,factuality_score,confidence_score,difficulty,language,source_type,created_year
0,lhdd2026-000001,At what temperature does water boil at standar...,Water boils at 100°C at standard atmospheric p...,Water boils at 100°C at standard atmospheric p...,faithful,none,science,fact_verification,yes,0.92,0.96,hard,English,synthetic,2026
1,lhdd2026-000002,At what temperature does water boil at standar...,Water boils at 100°C at standard atmospheric p...,Water boils at 120°C at standard atmospheric p...,hallucinated,factual_error,science,fact_verification,yes,0.40,0.95,hard,English,synthetic,2026
2,lhdd2026-000003,How long does Earth take to orbit the Sun?,The Earth orbits the Sun once approximately ev...,Earth takes approximately 365.25 days to orbit...,faithful,none,science,response_evaluation,yes,0.93,0.84,hard,English,synthetic,2026


In [3]:
df["label"].value_counts()

label
faithful        3369
hallucinated    1631
Name: count, dtype: int64

In [4]:
import re

def overlap(response, context):
    response_words = set(re.findall(r"[a-z0-9]+", str(response).lower()))
    context_words = set(re.findall(r"[a-z0-9]+", str(context).lower()))
    if not response_words:
        return 0
    return len(response_words & context_words) / len(response_words)

df["overlap"] = [overlap(r, c) for r, c in zip(df["response"], df["context"])]
df.groupby("label")["overlap"].mean()

label
faithful        0.936204
hallucinated    0.461354
Name: overlap, dtype: float64

In [5]:
df["prediction"] = df["overlap"].apply(lambda x: "hallucinated" if x < 0.6 else "faithful")
(df["prediction"] == df["label"]).mean()

np.float64(0.9312)

In [8]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, test_size=0.2, random_state=42)
(test["prediction"] == test["label"]).mean()

np.float64(0.93)

In [9]:
from sklearn.metrics import classification_report

print(classification_report(test["label"], test["prediction"]))

              precision    recall  f1-score   support

    faithful       0.90      1.00      0.95       661
hallucinated       1.00      0.79      0.88       339

    accuracy                           0.93      1000
   macro avg       0.95      0.90      0.92      1000
weighted avg       0.94      0.93      0.93      1000



In [10]:
misses = test[(test["label"] == "hallucinated") & (test["prediction"] == "faithful")]
print(len(misses))
misses[["prompt", "context", "response", "overlap"]].head(5)

70


,prompt,context,response,overlap
3543,Check the factual accuracy of this answer: Whe...,The Declaration of Independence was adopted by...,"It was adopted on July 4, 1789.",0.714286
3351,What is the capital of Japan?,Tokyo is the capital city of Japan.,Kyoto is currently the capital city of Japan.,0.750000
3862,Provide a grounded response to the question: W...,The Declaration of Independence was adopted by...,"It was adopted on July 4, 1789.",0.714286
3682,Answer using only the provided context: What i...,Tokyo is the capital city of Japan.,Kyoto is currently the capital city of Japan.,0.750000
296,At what temperature does water boil at standar...,Water boils at 100°C at standard atmospheric p...,Water boils at 120°C at standard atmospheric p...,0.875000


The missed answers repeat most of the context and change one fact, such as a city, a date, or a number. The dataset also repeats the same examples, so those rows show up more than once.